# ?? ???? ?? ??

- age: ??
- workclass: ?? ??
- fnlwgt: ??? ???? ???? ???(final weight)
- education: ?? ??
- education.num: ?? ?? ??
- marital.status: ?? ??
- occupation: ??
- relationship: ?? ??
- race: ??
- sex: ??
- capital.gain: ?? ??
- capital.loss: ?? ??
- hours.per.week: ?? ?? ??
- native.country: ??
- income: ?? (???? ?? ?)

In [ ]:
# ???? ?? (?? ?? X)
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

def exam_data_load(df, target, id_name="", null_name=""):
    if id_name == "":
        df = df.reset_index().rename(columns={"index": "id"})
        id_name = 'id'
    else:
        id_name = id_name
    
    if null_name != "":
        df[df == null_name] = np.nan
    
    X_train, X_test = train_test_split(df, test_size=0.2, random_state=2021)
    
    y_train = X_train[[id_name, target]]
    X_train = X_train.drop(columns=[target])

    
    y_test = X_test[[id_name, target]]
    X_test = X_test.drop(columns=[target])
    return X_train, X_test, y_train, y_test 
    
df = pd.read_csv("../input/adult-census-income/adult.csv")
X_train, X_test, y_train, y_test = exam_data_load(df, target='income', null_name='?')

X_train.shape, X_test.shape, y_train.shape, y_test.shape

# ??? ??

## ????? ????

In [ ]:
import pandas as pd
import numpy as np

## ??? ????(??)

In [ ]:
# ??????? ??? ?? ????? ?
# import pandas as pd
# X_test = pd.read_csv("data/X_test.csv")
# X_train = pd.read_csv("data/X_train.csv")
# y_train = pd.read_csv("data/y_train.csv")

## EDA

In [ ]:
# ??? ?? ??
X_train.shape, X_test.shape, y_train.shape

In [ ]:
# ??? ??
X_train.head()

In [ ]:
# ?? ? ??
y_train['income'].value_counts()

In [ ]:
# type??
X_train.info()

In [ ]:
# ?? ??
# Numeric features
numeric_features = [
                    'age',
                    'fnlwgt', 
                    'education.num',
                    'capital.gain', 
                    'capital.loss', 
                    'hours.per.week',                     
                   ]

# Categorical features
cat_features = [
                 'workclass',              
                 'education',            
                 'marital.status', 
                 'occupation', 
                 'relationship', 
                 'race', 
                 'sex',
                 'native.country'
]

In [ ]:
X_train[numeric_features].describe()

In [ ]:
X_train[cat_features].describe()

## ??? ??

In [ ]:
X_train.isnull().sum()

In [ ]:
X_test.isnull().sum()

In [ ]:
X_train['workclass'].value_counts()

In [ ]:
X_train['occupation'].value_counts()

In [ ]:
X_train['native.country'].value_counts()

- ???? ???? ??? ?? ????? ?? ???? ??? ??? ???

In [ ]:
def data_fillna(df):
    df['workclass'] = df['workclass'].fillna(df['workclass'].mode()[0])
    df['occupation'] = df['occupation'].fillna("null")
    df['native.country'] = df["native.country"].fillna(df['native.country'].mode()[0])
    return df

X_train = data_fillna(X_train)
X_test = data_fillna(X_test)

X_train.isnull().sum()

## ???????

In [ ]:
# ?????
from sklearn.preprocessing import LabelEncoder

all_df = pd.concat([X_train.assign(ind="train"), X_test.assign(ind="test")])
le = LabelEncoder()
all_df[cat_features] = all_df[cat_features].apply(le.fit_transform)

X_train = all_df[all_df['ind'] == 'train']
X_train = X_train.drop('ind',axis=1)
X_train

X_test = all_df[all_df['ind'] == 'test']
X_test = X_test.drop('ind',axis=1)
X_test

In [ ]:
# ????
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test[numeric_features] = scaler.transform(X_test[numeric_features])
X_train

In [ ]:
# target? ??
y = (y_train['income'] != '<=50K').astype(int)
y[:5]

## ??? ??? ??

In [ ]:
# ??? ???? ??? ???? ??
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y, test_size=0.15, random_state=2021)
X_tr.shape, X_val.shape, y_tr.shape, y_val.shape

In [ ]:
X_tr.head()

In [ ]:
# id ??
X_tr = X_tr.drop('id', axis=1)
X_val = X_val.drop('id', axis=1)

In [ ]:
# id ??? ? ??
X_tr.head(1)

## ?? & ??

In [ ]:
# ??????
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

model = DecisionTreeClassifier(random_state = 2022)
model.fit(X_tr, y_tr)
pred = model.predict(X_val)
print('accuracy score:', (accuracy_score(y_val, pred)))

In [ ]:
# ??????
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state = 2022)
model.fit(X_tr, y_tr)
pred = model.predict(X_val)
print('accuracy score:', (accuracy_score(y_val, pred)))

In [ ]:
# test??? ?? (pop? ???? ?? ???? ?? ?)
X_test_id = X_test.pop('id')
pred = model.predict(X_test)

In [ ]:
# csv??
output = pd.DataFrame({'id': X_test_id, 'income':pred})
output.to_csv("000000.csv", index=False)
output.head()

## ?? (???? ?? ??)

In [ ]:
y_test = (y_test['income'] != '<=50K').astype(int)
from sklearn.metrics import accuracy_score
print('accuracy score:', (accuracy_score(y_test, pred)))